In [1]:

# ── Configuration ─────────────────────────────────────────────────────────────
TARGET_LABELS = [
    "No_action", "កាតាប", "កាតាបស្ពាយក្រោយ", "កុំព្យូទ័រ", "កៅអី", "ក្ដារខៀន", "ខ្មៅដៃ",
    "ជ័រលុប", "ដីស", "តុ", "ទឹកលុប", "នាយករង", "នាយិកា",
    "បន្ទាត់", "ប៊ិក", "ប៊ិកក្រហម", "ប៊ិកខៀវ", "លោកគ្រូ", "សាលារៀន",
    "សៀវភៅ","ហ្វឺតក្រហម", "សៀវភៅពុម្ភ", "ហ្វឺតខៀវ", "ហ្វឺតខ្មៅ", "អ្នកគ្រូ"
]

# Feature dimension constants — change here propagates everywhere
POSE_FEATURES  = 33  * 4   # x, y, z, visibility
HAND_FEATURES  = 21  * 3   # x, y, z  (one hand)
EYE_INDICES = [
    33, 133, 157, 158, 159, 160, 161, 246, # Left Eye
    263, 362, 384, 385, 386, 387, 388, 466 # Right Eye
]
MOUTH_INDICES = [
    61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, # Outer Lips
    78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308, # Inner Lips
]
FACE_SELECTED_INDICES = EYE_INDICES + MOUTH_INDICES
FACE_FEATURES = len(FACE_SELECTED_INDICES) * 3 # 38 * 3 = 114 features

# Total position features per frame
POSITION_FEATURES = POSE_FEATURES + FACE_FEATURES + (HAND_FEATURES * 2)
# Calculation: 132 + 114 + (63 * 2) = 372

SEQUENCE_LENGTH = 30   # frames per training sample
USE_VELOCITY    = True  # append frame-delta features → doubles feature count

# Final feature count per frame
TOTAL_FEATURES = POSITION_FEATURES * 2 if USE_VELOCITY else POSITION_FEATURES
# With velocity: 372 * 2 = 744

# ── Sequence Index Generator ───────────────────────────────────────────────────
def get_sequence_indices(total_frames: int, seq_length: int, step: int = 15):
    """Generate overlapping window indices for sequence extraction"""
    indices = []
    
    if total_frames < seq_length:
        # Return single sequence (will be padded later)
        return [list(range(total_frames))]
    
    # Extract overlapping windows
    for start in range(0, total_frames - seq_length + 1, step):
        indices.append(list(range(start, start + seq_length)))
    
    # Add last window to capture ending
    last_start = total_frames - seq_length
    if last_start not in [idx[0] for idx in indices]:
        indices.append(list(range(last_start, last_start + seq_length)))
    
    return indices

def pad_sequence(frames, target_length=SEQUENCE_LENGTH):
    """Pad a sequence of frames to target length by repeating frames"""
    if len(frames) >= target_length:
        return frames
    
    # Calculate how many frames to pad
    pad_needed = target_length - len(frames)
    
    # Strategy: repeat frames evenly throughout the sequence
    if len(frames) == 1:
        # Just repeat the single frame
        padded = frames * target_length
    else:
        # Interpolate by repeating frames
        padded = []
        for i in range(target_length):
            # Map target index to original index
            orig_idx = int(i * len(frames) / target_length)
            orig_idx = min(orig_idx, len(frames) - 1)
            padded.append(frames[orig_idx].copy())
    
    return padded   
    

In [2]:
# ================================================================
# FEATURE CONFIG — change these to switch combinations
# ================================================================
USE_POSE     = False   # 33 points × 4 = 132
USE_FACE     = False   # selected points × 3 = 114 (set FACE_POINTS below)
USE_HANDS    = True    # 21 points × 3 × 2 hands = 126
USE_VELOCITY = True

# Face landmark indices (only used if USE_FACE = True)
FACE_SELECTED_INDICES = [
    61,146,91,181,84,17,314,405,321,375,   # mouth (10)
    291,308,324,318,402,317,14,87,178,88,  # mouth (10)
]  # 20 points × 3 = 60 features (Note: fixed comment arithmetic here)

FRAME_SKIP          = 2    # Process every 2nd frame (2x faster) - try 2 or 3
ROI_PADDING         = 0.15 # Reduced from 0.25 (40% smaller ROI)
HOLISTIC_CONF       = 0.4  # Reduced from 0.5 (faster)
HAND_CONF           = 0.6  # Reduced from 0.7 (faster)
MODEL_COMPLEXITY    = 1    # 0=fastest, 1=balanced, 2=accurate (was 1)
SEQUENCE_LENGTH     = 30   #  ADDED: Defined clearly so your print statement stays dynamic!
SEQUENCE_STEP_RATIO = 2    # Less overlap = fewer sequences (higher = fewer sequences)
NUM_AUGMENTATIONS   = 3    # Reduced from 5 (40% faster)
SKIP_IF_EXISTS      = True

# ── Auto-calculate feature sizes ─────────────────────────────
POSE_FEATURES  = 33 * 4 if USE_POSE  else 0   # 132
FACE_FEATURES  = len(FACE_SELECTED_INDICES) * 3 if USE_FACE else 0
HAND_FEATURES  = 63                             # per hand
HANDS_FEATURES = HAND_FEATURES * 2 if USE_HANDS else 0  # 126

POSITION_FEATURES = POSE_FEATURES + FACE_FEATURES + HANDS_FEATURES
TOTAL_FEATURES    = POSITION_FEATURES * 2 if USE_VELOCITY else POSITION_FEATURES

print("=" * 50)
print("FEATURE CONFIGURATION")
print("=" * 50)
print(f"USE_POSE     : {USE_POSE}  → {POSE_FEATURES} features")
print(f"USE_FACE     : {USE_FACE}  → {FACE_FEATURES} features")
print(f"USE_HANDS    : {USE_HANDS}  → {HANDS_FEATURES} features")
print(f"USE_VELOCITY : {USE_VELOCITY}")
print(f"─────────────────────────────")
print(f"Position features : {POSITION_FEATURES}")
print(f"Total features    : {TOTAL_FEATURES}")
print(f"Model input shape : ({SEQUENCE_LENGTH}, {TOTAL_FEATURES})") # ✨ FIXED: Uses variable instead of hardcoded 30
print("=" * 50)

FEATURE CONFIGURATION
USE_POSE     : False  → 0 features
USE_FACE     : False  → 0 features
USE_HANDS    : True  → 126 features
USE_VELOCITY : True
─────────────────────────────
Position features : 126
Total features    : 252
Model input shape : (30, 252)


In [3]:
class Hand_Pose_Processor:
    _WRIST_LEFT  = 15
    _WRIST_RIGHT = 16
    _SHOULDER_LEFT = 11
    _SHOULDER_RIGHT = 12

    def __init__(self, roi_padding=0.15, holistic_conf=0.4, hand_conf=0.6):
        self.roi_padding = roi_padding
        
        self._holistic = mp.solutions.holistic.Holistic(
            static_image_mode=False,
            model_complexity=MODEL_COMPLEXITY,
            min_detection_confidence=holistic_conf,
            min_tracking_confidence=holistic_conf,
            enable_segmentation=False,
        )
        self._hands = mp.solutions.hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            model_complexity=MODEL_COMPLEXITY,
            min_detection_confidence=hand_conf,
            min_tracking_confidence=0.5,
        )

    def _to_rgb(self, frame):
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    def _get_roi(self, frame, wrist_lm):
        h, w = frame.shape[:2]
        cx = int(wrist_lm.x * w)
        cy = int(wrist_lm.y * h)
        box = int(h * self.roi_padding)
        x1 = max(0, cx - box // 2)
        y1 = max(0, cy - box // 2)
        x2 = min(w, x1 + box)
        y2 = min(h, y1 + box)
        return frame[y1:y2, x1:x2], (x1, y1)

    def _extract_roi_hand(self, frame, wrist_lm):
        roi, (ox, oy) = self._get_roi(frame, wrist_lm)
        if roi.size == 0:
            return np.zeros(HAND_FEATURES, dtype=np.float32)

        h_g, w_g = frame.shape[:2]
        h_r, w_r = roi.shape[:2]
        
        if h_r < 20 or w_r < 20:
            return np.zeros(HAND_FEATURES, dtype=np.float32)
            
        result = self._hands.process(self._to_rgb(roi))

        if not result.multi_hand_landmarks:
            return np.zeros(HAND_FEATURES, dtype=np.float32)

        pts = []
        for lm in result.multi_hand_landmarks[0].landmark:
            pts.extend([
                (lm.x * w_r + ox) / w_g,
                (lm.y * h_r + oy) / h_g,
                lm.z
            ])
        arr = np.array(pts, dtype=np.float32)
        wrist_offset = arr[:3].copy()
        for i in range(0, len(arr), 3):
            arr[i:i+3] -= wrist_offset
        return arr

    def extract_all_features(self, frame_bgr):
        rgb = self._to_rgb(frame_bgr)
        result = self._holistic.process(rgb)
        
        left_hand = np.zeros(HAND_FEATURES, dtype=np.float32)
        right_hand = np.zeros(HAND_FEATURES, dtype=np.float32)
        pose_data = np.zeros(POSE_FEATURES, dtype=np.float32)

        # 1. PROCESS POSE AND EXTRACT BODY LANDMARKS
        if result.pose_landmarks:
            lms = result.pose_landmarks.landmark
            
            # Establish Mid-Neck/Shoulder Anchor point for body normalization
            ls_x, ls_y, ls_z = lms[self._SHOULDER_LEFT].x, lms[self._SHOULDER_LEFT].y, lms[self._SHOULDER_LEFT].z
            rs_x, rs_y, rs_z = lms[self._SHOULDER_RIGHT].x, lms[self._SHOULDER_RIGHT].y, lms[self._SHOULDER_RIGHT].z
            
            neck_anchor = np.array([(ls_x + rs_x) / 2.0, (ls_y + rs_y) / 2.0, (ls_z + rs_z) / 2.0], dtype=np.float32)
            
            # Format and normalize pose points relative to the neck anchor
            raw_pose = []
            for l in lms:
                # Subtract anchor from x, y, z positions; preserve raw visibility score
                raw_pose.extend([l.x - neck_anchor[0], l.y - neck_anchor[1], l.z - neck_anchor[2], l.visibility])
            pose_data = np.array(raw_pose, dtype=np.float32)
            
            # 2. RUN HIGH-RESOLUTION ROI HAND DETECTION USING POSE WRIST ANCHORS
            lw = lms[self._WRIST_LEFT]
            if lw.visibility > 0.3:
                left_hand = self._extract_roi_hand(frame_bgr, lw)
                if np.all(left_hand == 0) and result.left_hand_landmarks:
                    pts = np.array([[l.x, l.y, l.z] for l in result.left_hand_landmarks.landmark], dtype=np.float32)
                    left_hand = (pts - pts[0]).flatten()
            
            rw = lms[self._WRIST_RIGHT]
            if rw.visibility > 0.3:
                right_hand = self._extract_roi_hand(frame_bgr, rw)
                if np.all(right_hand == 0) and result.right_hand_landmarks:
                    pts = np.array([[l.x, l.y, l.z] for l in result.right_hand_landmarks.landmark], dtype=np.float32)
                    right_hand = (pts - pts[0]).flatten()
        else:
            # Fallback if the body isn't detected at all, tracking only hands
            if result.left_hand_landmarks:
                pts = np.array([[l.x, l.y, l.z] for l in result.left_hand_landmarks.landmark], dtype=np.float32)
                left_hand = (pts - pts[0]).flatten()
            if result.right_hand_landmarks:
                pts = np.array([[l.x, l.y, l.z] for l in result.right_hand_landmarks.landmark], dtype=np.float32)
                right_hand = (pts - pts[0]).flatten()
        
        # 3. CONCATENATE ALL POSITION MATRIX CHUNKS
        return np.concatenate([pose_data, left_hand, right_hand])

    def close(self):
        self._holistic.close()
        self._hands.close()

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp
from pathlib import Path
from tqdm import tqdm

# ============================================================
# CONFIGURATION AND PARAMETER INITIALIZATION
# ============================================================
BASE_DIR = Path(r"D:\KSLA-DP\ksl")
DATA_DIR = BASE_DIR / 'education words' / '24_sign_SHORT_FRAMES'
LANDMARK_DIR = BASE_DIR / 'landmarks_30frames_short' # Updated folder name to match target

SEQUENCE_LENGTH = 30
SKIP_IF_EXISTS  = True

USE_HANDS       = True
USE_VELOCITY    = True

POSE_FEATURES   = 132 
HAND_FEATURES   = 63  

BASE_POSITIONS = 0
if USE_POSE:  BASE_POSITIONS += POSE_FEATURES
if USE_HANDS: BASE_POSITIONS += (HAND_FEATURES * 2)

TOTAL_FEATURES = BASE_POSITIONS * 2 if USE_VELOCITY else BASE_POSITIONS

TARGET_LABELS = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])

# ============================================================
# NATIVE VECTOR EXTRACTION HELPERS
# ============================================================
def extract_native_hand(hand_landmarks):
    if not hand_landmarks:
        return np.zeros(HAND_FEATURES, dtype=np.float32)
    pts = []
    for lm in hand_landmarks.landmark:
        pts.extend([lm.x, lm.y, lm.z])
    arr = np.array(pts, dtype=np.float32).reshape(-1, 3)
    arr -= arr[0].copy() 
    return arr.flatten()

print("=" * 60)
print("🚀 STARTING TRUE SEQUENTIAL EXTRACTION ENGINE (SHORT TRACK)")
print(f"Target Dimensions: ({SEQUENCE_LENGTH}, {TOTAL_FEATURES})")
print("=" * 60)

mp_holistic = mp.solutions.holistic

for class_name in TARGET_LABELS:
    class_path = DATA_DIR / class_name
    save_path = LANDMARK_DIR / class_name
    
    if SKIP_IF_EXISTS and save_path.exists():
        existing_samples = list(save_path.glob("*.npy"))
        if len(existing_samples) > 0:
            print(f"⏭️ Skipping Class: {class_name:<20s} | Already processed ({len(existing_samples)} samples found)")
            continue
            
    save_path.mkdir(parents=True, exist_ok=True)
    video_files = list(class_path.glob("*.mp4"))
    print(f"\nProcessing {class_name}: {len(video_files)} videos")
    
    sample_counter = 0
    
    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1, 
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:
        
        for video_file in tqdm(video_files, desc=class_name):
            cap = cv2.VideoCapture(str(video_file))
            
            all_video_features = []
            
            # ── 1. ONE-PASS SEQUENTIAL INFRASTRUCTURE ───────────────────────
            while True:
                ret, frame = cap.read()
                if not ret: break
                
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = holistic.process(rgb)
                
                frame_features = []
                
                # Pose Extraction
                if USE_POSE:
                    if results.pose_landmarks:
                        lms = results.pose_landmarks.landmark
                        ls, rs = lms[11], lms[12]
                        anchor_x = (ls.x + rs.x) / 2.0
                        anchor_y = (ls.y + rs.y) / 2.0
                        anchor_z = (ls.z + rs.z) / 2.0
                        
                        raw_pose = []
                        for l in lms:
                            raw_pose.extend([l.x - anchor_x, l.y - anchor_y, l.z - anchor_z, l.visibility])
                        pose_pts = np.array(raw_pose, dtype=np.float32)
                    else:
                        pose_pts = np.zeros(POSE_FEATURES, dtype=np.float32)
                    frame_features.append(pose_pts)
                    
                # Hand Extraction
                if USE_HANDS:
                    left_hand = extract_native_hand(results.left_hand_landmarks)
                    right_hand = extract_native_hand(results.right_hand_landmarks)
                    frame_features.extend([left_hand, right_hand])
                
                combined_positions = np.concatenate(frame_features)
                all_video_features.append(combined_positions)
                
            cap.release()
            
            total_frames = len(all_video_features)
            if total_frames == 0: continue 
            
            # ── 2. STANDARDIZE TIMELINE TO EXACT SEQUENCE LENGTH ─────────────────
            all_video_features = np.array(all_video_features, dtype=np.float32)
            
            if total_frames >= SEQUENCE_LENGTH:
                # Video has enough frames: Select a uniform sample spread evenly across the clip
                indices = np.linspace(0, total_frames - 1, SEQUENCE_LENGTH, dtype=int)
                positions = all_video_features[indices]
            else:
                # Video is too short: Grab everything and pad missing frames using the last frame's posture
                pad_needed = SEQUENCE_LENGTH - total_frames
                last_frame = all_video_features[-1:] 
                padding = np.repeat(last_frame, pad_needed, axis=0)
                positions = np.concatenate([all_video_features, padding], axis=0)
            
            # ── 3. DYNAMIC VELOCITY CALCULATION ──────────────────────────────
            if USE_VELOCITY:
                velocity = np.zeros_like(positions)
                velocity[1:] = positions[1:] - positions[:-1]
                full_features = np.concatenate([positions, velocity], axis=1)
            else:
                full_features = positions
            
            # Safety verification check
            assert full_features.shape == (SEQUENCE_LENGTH, TOTAL_FEATURES), \
                f"Shape conflict! Got {full_features.shape}, expected ({SEQUENCE_LENGTH}, {TOTAL_FEATURES})"
            
            # Save final matrix directly to disk
            save_file = save_path / f"{sample_counter:06d}.npy"
            np.save(save_file, full_features)
            sample_counter += 1
                
    print(f"  Created {sample_counter} samples from {len(video_files)} videos")

print("\n" + "=" * 60)
print("🏁 EXTRACTION COMPLETE")
print("=" * 60)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'D:\\uni\\Intern-1-Project\\ksl\\education words\\24_sign_SHORT_FRAMES'